In [2]:
# first we go through data augmentation and transforming the augmentated data into a pickle format 

#link to the data 


https://www.kaggle.com/datasets/tauilabdelilah/mrl-eye-dataset

In [2]:
import os
import cv2
import numpy as np
from tqdm import tqdm

In [3]:
# STEP 1 — CONFIGURATION
# Define where our data lives and what size we want images to be
# We use 32x32 because:
#   - Small enough to run fast on edge devices
#   - Large enough to capture eye open/closed features
# ────────────

In [3]:
data_dir = 'data/train'

img_size = 32

CLASSES = {
    'open' : 1,
    'closed' : 0
}

In [4]:
# ─────────────────────────────────────────────────────────────
# STEP 2 — LOAD A SINGLE IMAGE AND CONVERT TO GRAYSCALE
# Why grayscale?
#   - Eye open/closed detection does not need color information
#   - 1 channel instead of 3 → smaller model, faster inference
#   - Better for IR camera output (which is already grayscale)

In [5]:
def load_image(path):
    img = cv2.imread(path) #load as rgb default

    if img is None:
        return None #skip corrupted
    gray = cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)

    resized = cv2.resize(gray,(img_size,img_size))

    return resized

In [6]:
# ─────────────────────────────────────────────────────────────
# STEP 3 — AUGMENTATION PIPELINE
# Why augment?
#   - 30k images is good but augmentation adds variety
#   - Teaches the model to handle real-world conditions:
#     different angles, lighting, blur, etc.
#   - Prevents overfitting
#
# Each image produces 8 variants including the original

In [7]:
def augment(img):
    augmented = []
     # ── 3a. Original image (no change) ──────────────────────
    augmented.append(img)
    # ── 3b. Horizontal flip ──────────────────────────────────
    # Simulates left/right eye symmetry
    flipped = cv2.flip(img, 1)
    augmented.append(flipped)
 
    # ── 3c. Rotation ±15 degrees ────────────────────────────
    # Simulates head tilt — common when drowsy

    h,w = img.shape
    center = (w //2, h // 2)

    for angle in [-15,15]:
        rotation_matrix = cv2.getRotationMatrix2D(center,angle,1.0)

        rotated = cv2.warpAffine(img,rotation_matrix,(w,h))

        augmented.append(rotated)

    # ── 3d. Gaussian blur ────────────────────────────────────
    # Simulates motion blur or out-of-focus camera

    blurred = cv2.GaussianBlur(img,(3,3),0)
    augmented.append(blurred)

    # ── 3e. Brightness increase ──────────────────────────────
    # Simulates bright environment / sunlight
    bright = np.clip(img.astype(np.int32) + 40,0,255).astype(np.uint8)
    augmented.append(bright)


    # ── 3f. Brightness decrease ──────────────────────────────
    # Simulates dim environment / night driving
    dark = np.clip(img.astype(np.int32) -40,0,255).astype(np.uint8)
    augmented.append(dark)

     # ── 3g. Edge detection (Canny) ───────────────────────────
    # Highlights eye contour and eyelid edges
    # Helps the model learn structural features of open vs closed

    edges = cv2.Canny(img,50,150)
    augmented.append(edges)

    # Returns 8 variants for every single input image
    return augmented
 

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 4 — PROCESS THE FULL DATASET
# Loop through every image in open/ and closed/ folders
# Apply load + augment pipeline to each
# ─────────────────────────────────────────────────────────────

In [9]:
def process_dataset():
    X = []   # will hold all image arrays
    y = []   # will hold all labels (0 or 1)
 
    for label_name, label_value in CLASSES.items():
        folder = os.path.join(data_dir, label_name)
        files  = os.listdir(folder)
 
        print(f"\n── Processing class: [{label_name}] ──")
        print(f"   Images found : {len(files)}")
        print(f"   After augment: {len(files) * 8} (8x each)")
 
        for fname in tqdm(files, desc=label_name):
            img_path = os.path.join(folder, fname)
 
            # STEP 4a — load and convert to grayscale 32x32
            img = load_image(img_path)
            if img is None:
                continue
 
            # STEP 4b — generate 8 augmented variants
            variants = augment(img)
 
            for variant in variants:
                # STEP 4c — normalize pixel values from [0,255] → [0.0, 1.0]
                # Neural networks train better with small input values
                normalized = variant.astype(np.float32) / 255.0
 
                # STEP 4d — add channel dimension (32,32) → (32,32,1)
                # Keras expects shape (height, width, channels)
                final = normalized.reshape(img_size, img_size, 1)
 
                X.append(final)
                y.append(label_value)
 
    # STEP 4e — convert lists to numpy arrays
    X = np.array(X)   # shape: (total_samples, 32, 32, 1)
    y = np.array(y)   # shape: (total_samples,)
 
    return X, y

In [10]:
# ─────────────────────────────────────────────────────────────
# STEP 5 — RUN AND PREVIEW RESULTS
# ─────────────────────────────────────────────────────────────

In [11]:
print("Starting data processing pipeline..\n")

X,y = process_dataset()

# Shuffle so open/closed are mixed (not all open first, then all closed)
print("\nShuffling dataset...")
idx  = np.random.permutation(len(X))
X, y = X[idx], y[idx]

Starting data processing pipeline..


── Processing class: [open] ──
   Images found : 106482
   After augment: 851856 (8x each)


open: 100%|██████████| 106482/106482 [01:32<00:00, 1148.04it/s]



── Processing class: [closed] ──
   Images found : 33322
   After augment: 266576 (8x each)


closed: 100%|██████████| 33322/33322 [00:21<00:00, 1533.22it/s]



Shuffling dataset...


In [12]:
# ─────────────────────────────────────────────────────────────
# STEP 6 — SUMMARY
# ─────────────────────────────────────────────────────────────

In [ ]:
print("\n── Dataset Summary ──────────────────────────")
print(f"  Total samples : {len(X)}")
print(f"  X shape       : {X.shape}  →  (samples, 32, 32, 1)")
print(f"  y shape       : {y.shape}")
print(f"  Open  (1)     : {np.sum(y == 1)}")
print(f"  Closed (0)    : {np.sum(y == 0)}")
print(f"  Pixel range   : {X.min():.2f} → {X.max():.2f}  (normalized)")
print("─────────────────────────────────────────────")
print("\nReady for pickle export.")


── Dataset Summary ──────────────────────────
  Total samples : 1118432
  X shape       : (1118432, 32, 32, 1)  →  (samples, 32, 32, 1)
  y shape       : (1118432,)
  Open  (1)     : 851856
  Closed (0)    : 266576
  Pixel range   : 0.00 → 1.00  (normalized)
─────────────────────────────────────────────

Ready for pickle export.


In [14]:
# Add this after the shuffle, before pickle export

# Match open count to closed count
closed_idx = np.where(y == 0)[0]               # all closed indices
open_idx   = np.where(y == 1)[0]               # all open indices

# Randomly pick same number of open samples as closed
open_idx_sampled = np.random.choice(open_idx, size=len(closed_idx), replace=False)

# Combine and shuffle again
balanced_idx = np.concatenate([closed_idx, open_idx_sampled])
np.random.shuffle(balanced_idx)

X = X[balanced_idx]
y = y[balanced_idx]

print(f"After balancing:")
print(f"  Open  (1) : {np.sum(y == 1)}")
print(f"  Closed (0): {np.sum(y == 0)}")
print(f"  Total     : {len(X)}")

After balancing:
  Open  (1) : 266576
  Closed (0): 266576
  Total     : 533152


In [15]:
#compress to pickle

import pickle

OUTPUT = "eye_dataset.pkl"

print(f"\nSaving to {OUTPUT}...")
with open(OUTPUT, "wb") as f:
    pickle.dump({"X": X, "y": y}, f)

print(f"Done. File size: {os.path.getsize(OUTPUT) / 1e6:.1f} MB")


Saving to eye_dataset.pkl...
Done. File size: 2188.1 MB


In [16]:
#now lets train 

In [17]:
import pickle
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split

I0000 00:00:1779469601.420022    4411 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [18]:
# ─────────────────────────────────────────────────────────────
# STEP 1 — LOAD PICKLE
# Load the preprocessed and balanced dataset we saved earlier
# ─────────────────────────────────────────────────────────────

In [19]:
print("Loading dataset...")
with open("eye_dataset.pkl", "rb") as f:
    data = pickle.load(f)

X = data["X"] # shape: (533152, 32, 32, 1)
y = data["y"]   # shape: (533152,)

print(f"  X shape : {X.shape}")
print(f"  y shape : {y.shape}")
print(f"  Open    : {np.sum(y == 1)}")
print(f"  Closed  : {np.sum(y == 0)}")

Loading dataset...
  X shape : (533152, 32, 32, 1)
  y shape : (533152,)
  Open    : 266576
  Closed  : 266576


In [20]:
# ─────────────────────────────────────────────────────────────
# STEP 2 — TRAIN / VALIDATION SPLIT
# 80% for training, 20% for validation
# Validation tells us how well the model generalises
# to images it has never seen during training
# ─────────────────────────────────────────────────────────────

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,       # 20% validation
    random_state=42,     # reproducible split
    stratify=y           # keeps open/closed ratio equal in both splits
)
 
print(f"\nTrain samples : {len(X_train)}")
print(f"Val   samples : {len(X_val)}")


Train samples : 426521
Val   samples : 106631


In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 3 — BUILD THE CNN MODEL
#
# Architecture (simple and edge-friendly):
#
#   Input (32, 32, 1)
#     ↓
#   Conv2D 32 filters → learns basic edges and shapes
#     ↓
#   MaxPooling → reduces size, keeps strongest features
#     ↓
#   Conv2D 64 filters → learns more complex patterns
#     ↓
#   MaxPooling
#     ↓
#   Conv2D 128 filters → learns high-level eye features
#     ↓
#   GlobalAveragePooling → flattens without large dense layer
#     ↓
#   Dropout 0.5 → randomly drops neurons to prevent overfitting
#     ↓
#   Dense 1 + Sigmoid → outputs probability (0=closed, 1=open)
# ─────────────────────────────────────────────────────────────

Block 1 output (32x32, before pooling):

  Each unit sees only a 3x3 patch of the RAW pixels
 
  → can only detect: edges, gradients, local contrast
 
  → physically CANNOT see anything bigger than 3x3

After MaxPool → 16x16:
 
  Each unit now summarizes a 2x2 area of Block1's output
 
  → indirectly "sees" roughly a 4x4-6x6 patch of the ORIGINAL image

Block 2 output (16x16, before its pooling):
 
  Each unit's 3x3 filter operates on Block1's (pooled) output
 
  → effective view of original image grows further

After MaxPool → 8x8:
 
  → each unit now "sees" roughly an 8x8-10x10 patch of original

Block 3 output (8x8, before its pooling):
 
  → effective view grows again

After MaxPool → 4x4:
 
  → each unit now "sees" a large portion of the ENTIRE 32x32 crop

In [ ]:
model = keras.Sequential([
 
    # Input layer — tell Keras what shape each sample is
    keras.Input(shape=(32, 32, 1)),
 
    # Block 1 — detect basic edges and gradients
    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D(2, 2),          # 32x32 → 16x16
 
    # Block 2 — detect eye region patterns
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D(2, 2),          # 16x16 → 8x8
 
    # Block 3 — detect high level open/closed features
    layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D(2, 2),          # 8x8 → 4x4
 
    # Flatten feature maps into a single vector
    layers.GlobalAveragePooling2D(),    # 4x4x128 → 128
 
    # Dropout — prevents overfitting by randomly zeroing 50% of neurons
    layers.Dropout(0.5),
 
    # Output — single neuron, sigmoid gives probability between 0 and 1
    # > 0.5 → open (1),  < 0.5 → closed (0)
    layers.Dense(1, activation="sigmoid")
])
 
model.summary()

E0000 00:00:1779469933.122766    4411 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 92,801 (362.50 KB)

 Trainable params: 92,801 (362.50 KB)

 Non-trainable params: 0 (0.00 B)

In [24]:
# ─────────────────────────────────────────────────────────────
# STEP 4 — COMPILE THE MODEL
# optimizer : Adam adjusts learning rate automatically
# loss      : binary_crossentropy for open/closed classification
# metrics   : accuracy tells us % correct predictions
# ─────────────────────────────────────────────────────────────


In [25]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [26]:
# ─────────────────────────────────────────────────────────────
# STEP 5 — CALLBACKS
# These monitor training and act automatically:
#
# EarlyStopping  — stops training if val_loss stops improving
#                  prevents wasted time and overfitting
# ModelCheckpoint — saves the best model weights automatically
# ─────────────────────────────────────────────────────────────

In [27]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,           # stop if no improvement for 5 epochs
        restore_best_weights=True
    ),
    keras.callbacks.ModelCheckpoint(
        filepath="eye_model_best.h5",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    )
]

In [28]:
# ─────────────────────────────────────────────────────────────
# STEP 6 — TRAIN
# batch_size=64  : process 64 images at a time
# epochs=30      : max 30 passes through the dataset
#                  EarlyStopping will likely stop it earlier
# ─────────────────────────────────────────────────────────────

In [ ]:
print("\nStarting training...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=64,
    epochs=30,
    callbacks=callbacks,
    verbose=1
)


Starting training...
Epoch 1/30


W0000 00:00:1779470006.076444    4411 cpu_allocator_impl.cc:82] Allocation of 1747030016 exceeds 10% of free system memory.


6663/6665 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.8597 - loss: 0.3333

W0000 00:00:1779470194.996351    4411 cpu_allocator_impl.cc:82] Allocation of 436760576 exceeds 10% of free system memory.



Epoch 1: val_accuracy improved from None to 0.95031, saving model to eye_model_best.h5


...


Epoch 15: finished saving model to eye_model_best.h5
6665/6665 ━━━━━━━━━━━━━━━━━━━━ 195s 29ms/step - accuracy: 0.9780 - loss: 0.0581 - val_accuracy: 0.9778 - val_loss: 0.0616
Epoch 16/30
6663/6665 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.9787 - loss: 0.0569
Epoch 16: val_accuracy improved from 0.97779 to 0.97827, saving model to eye_model_best.h5



Epoch 16: finished saving model to eye_model_best.h5
6665/6665 ━━━━━━━━━━━━━━━━━━━━ 194s 29ms/step - accuracy: 0.9788 - loss: 0.0566 - val_accuracy: 0.9783 - val_loss: 0.0617
Epoch 17/30
6663/6665 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.9789 - loss: 0.0565
Epoch 17: val_accuracy improved from 0.97827 to 0.97848, saving model to eye_model_best.h5



Epoch 17: finished saving model to eye_model_best.h5
6665/6665 ━━━━━━━━━━━━━━━━━━━━ 194s 29ms/step - accuracy: 0.9789 - loss: 0.0564 - val_accuracy: 0.9785 - val_loss: 0.0613
Epoch 18/30
6664/6665 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.9797 - loss: 0.0546
Epoch 18: val_accuracy improved from 0.97848 to 0.97873, saving model to eye_model_best.h5



Epoch 18: finished saving model to eye_model_best.h5
6665/6665 ━━━━━━━━━━━━━━━━━━━━ 194s 29ms/step - accuracy: 0.9795 - loss: 0.0548 - val_accuracy: 0.9787 - val_loss: 0.0577
Epoch 19/30
6663/6665 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.9806 - loss: 0.0522
Epoch 19: val_accuracy improved from 0.97873 to 0.97883, saving model to eye_model_best.h5



Epoch 19: finished saving model to eye_model_best.h5
6665/6665 ━━━━━━━━━━━━━━━━━━━━ 194s 29ms/step - accuracy: 0.9801 - loss: 0.0537 - val_accuracy: 0.9788 - val_loss: 0.0595
Epoch 20/30
6663/6665 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.9805 - loss: 0.0523
Epoch 20: val_accuracy did not improve from 0.97883
6665/6665 ━━━━━━━━━━━━━━━━━━━━ 196s 29ms/step - accuracy: 0.9801 - loss: 0.0528 - val_accuracy: 0.9788 - val_loss: 0.0586
Epoch 21/30
6663/6665 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.9807 - loss: 0.0512
Epoch 21: val_accuracy did not improve from 0.97883
6665/6665 ━━━━━━━━━━━━━━━━━━━━ 197s 30ms/step - accuracy: 0.9803 - loss: 0.0520 - val_accuracy: 0.9788 - val_loss: 0.0612
Epoch 22/30
6663/6665 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.9811 - loss: 0.0506
Epoch 22: val_accuracy did not improve from 0.97883
6665/6665 ━━━━━━━━━━━━━━━━━━━━ 196s 29ms/step - accuracy: 0.9808 - loss: 0.0512 - val_accuracy: 0.9778 - val_loss: 0.0611
Epoch 23/30
6663/6665 ━━━━━━━━━━

In [31]:
# ─────────────────────────────────────────────────────────────
# STEP 7 — EVALUATE ON VALIDATION SET
# Final check — how well does the model perform overall
# ─────────────────────────────────────────────────────────────

In [32]:
val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
print(f"\nValidation Accuracy : {val_acc * 100:.2f}%")
print(f"Validation Loss     : {val_loss:.4f}")

W0000 00:00:1779474679.903212    4411 cpu_allocator_impl.cc:82] Allocation of 436760576 exceeds 10% of free system memory.



Validation Accuracy : 97.87%
Validation Loss     : 0.0577


In [33]:
# ─────────────────────────────────────────────────────────────
# STEP 8 — SAVE MODEL
# Save as .h5 for reference and reloading in Keras
# ─────────────────────────────────────────────────────────────

In [34]:
model.save("eye_model.h5")
print("\nSaved → eye_model.h5")


Saved → eye_model.h5


In [35]:
# ─────────────────────────────────────────────────────────────
# STEP 9 — CONVERT TO TFLITE (ai-edge-litert compatible)
# TFLite is the lightweight format for edge deployment
# INT8 quantization shrinks the model further for Pi/Jetson
# ─────────────────────────────────────────────────────────────

In [37]:
print("\nConverting to TFLite...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
# Optimize for size and speed on edge devices
converter.optimizations = [tf.lite.Optimize.DEFAULT]
 
tflite_model = converter.convert()
 
with open("eye_model.tflite", "wb") as f:
    f.write(tflite_model)


Converting to TFLite...
INFO:tensorflow:Assets written to: /tmp/tmpzgrn7ma8/assets


INFO:tensorflow:Assets written to: /tmp/tmpzgrn7ma8/assets


Saved artifact at '/tmp/tmpzgrn7ma8'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 32, 32, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  132423474581136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132430635367312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132423474581520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132423474581328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132423474583824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132423474585168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132423474583440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132423474583248: TensorSpec(shape=(), dtype=tf.resource, name=None)


W0000 00:00:1779474814.130562    4411 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1779474814.130588    4411 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1779474814.134719    4411 reader.cc:83] Reading SavedModel from: /tmp/tmpzgrn7ma8
I0000 00:00:1779474814.135156    4411 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1779474814.135161    4411 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpzgrn7ma8
I0000 00:00:1779474814.142992    4411 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1779474814.144334    4411 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1779474814.163410    4411 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpzgrn7ma8
I0000 00:00:1779474814.169680    4411 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 34971 microseconds.
I0000 00:00:1779474814.238609    4411

In [38]:
import os
size_kb = os.path.getsize("eye_model.tflite") / 1024
print(f"Saved → eye_model.tflite  ({size_kb:.1f} KB)")
print("\nDone. Ready for edge deployment.")
 

Saved → eye_model.tflite  (98.8 KB)

Done. Ready for edge deployment.


In [1]:
import mediapipe as mp
from ai_edge_litert.interpreter import Interpreter
print(mp.__version__)
print("all good")

I0000 00:00:1779524261.063337   15384 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


0.10.35
all good


In [1]:
#rcNN

In [ ]:

#MobileNetV2 expects values in range [-1, 1] not [0, 1]:

In [2]:
import cv2
import numpy as np
import os

def load_image(path):
    img = cv2.imread(path)
    if img is None:
        return None
    img = cv2.resize(img, (96, 96))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

def preprocess(img):
    # convert to float32
    img = img.astype(np.float32)
    # MobileNetV2 normalisation: maps 0-255 → -1.0 to +1.0
    img = (img / 127.5) - 1.0
    return img

# automatically grab first image from open folder
sample_path = os.path.join("data/train/open", os.listdir("data/train/open")[0])

img     = load_image(sample_path)
img_pre = preprocess(img)

print(f"Path  : {sample_path}")
print(f"Shape : {img_pre.shape}")
print(f"Range : {img_pre.min().round(3)} to {img_pre.max().round(3)}")

Path  : data/train/open/kaggle_flickrfaceshq-dataset-ffhq_41599_eye_LEFT.png
Shape : (96, 96, 3)
Range : -1.0 to 0.3019999861717224


In [3]:
#Step 3 — first augmentation: horizontal flip

def augment(img):
    variants = []

    #origninal
    variants.append(img)

    #horizintal flip simulate left/right eye symmetry

    flipped = cv2.flip(img,1)
    variants.append(flipped)
    return variants

#test it 
img = load_image(sample_path)
variants = augment(img)

print(f"Variants generated: {len(variants)}")
print(f"each shape:{variants[0].shape}")

Variants generated: 2
each shape:(96, 96, 3)


In [4]:
#Now add rotation inside the augment function:

def augment(img):
    variants = []

    h,w = img.shape[:2]

    centre = (w // 2,h // 2)

    #1 original
    variants.append(img)

    #2 horizontal flip
    variants.append(cv2.flip(img,1))

    #3 rotate +15 degree
    M = cv2.getRotationMatrix2D(centre,15,1.0)
    variants.append(cv2.warpAffine(img,M,(w,h)))

    #4 rotate -15 degree
    M = cv2.getRotationMatrix2D(centre, -15,1.0)
    variants.append(cv2.warpAffine(img,M,(w,h)))

     # 5. brightness increase
    # simulates bright sunlight
    bright = np.clip(img.astype(np.int32) + 40, 0, 255).astype(np.uint8)
    variants.append(bright)

    # 6. brightness decrease
    # simulates night / dim environment
    dark = np.clip(img.astype(np.int32) - 40, 0, 255).astype(np.uint8)
    variants.append(dark)


      # 7. gaussian blur
    # simulates motion blur or out of focus camera
    blurred = cv2.GaussianBlur(img, (5, 5), 0)
    variants.append(blurred)

    # 8. sharpen
    # simulates high contrast IR camera
    kernel  = np.array([[0, -1, 0],
                        [-1, 5,-1],
                        [0, -1, 0]])
    sharpened = cv2.filter2D(img, -1, kernel)
    variants.append(sharpened)

    # 9. random noise
    # simulates IR camera sensor noise
    noise     = np.random.randint(0, 25, img.shape, dtype=np.uint8)
    noisy     = cv2.add(img, noise)
    variants.append(noisy)

    # 10. random erasing
    # simulates glasses frame blocking part of eye
    erased    = img.copy()
    x1        = np.random.randint(0, 60)
    y1        = np.random.randint(0, 60)
    x2        = x1 + np.random.randint(20, 40)
    y2        = y1 + np.random.randint(20, 40)
    erased[y1:y2, x1:x2] = 0
    variants.append(erased)

    # 11. zoom in (1.2x)
    # simulates driver closer to camera
    zoom_factor = 1.2
    zh          = int(h / zoom_factor)
    zw          = int(w / zoom_factor)
    top         = (h - zh) // 2
    left        = (w - zw) // 2
    cropped     = img[top:top+zh, left:left+zw]
    zoomed      = cv2.resize(cropped, (w, h))
    variants.append(zoomed)

    # 12. elastic distortion
    # simulates head movement blur
    alpha       = 20
    sigma       = 5
    dx          = cv2.GaussianBlur(
                    (np.random.rand(h, w) * 2 - 1).astype(np.float32),
                    (0, 0), sigma) * alpha
    dy          = cv2.GaussianBlur(
                    (np.random.rand(h, w) * 2 - 1).astype(np.float32),
                    (0, 0), sigma) * alpha
    x, y        = np.meshgrid(np.arange(w), np.arange(h))
    map_x       = np.clip(x + dx, 0, w - 1).astype(np.float32)
    map_y       = np.clip(y + dy, 0, h - 1).astype(np.float32)
    distorted   = cv2.remap(img, map_x, map_y, cv2.INTER_LINEAR)
    variants.append(distorted)

    return variants

#test
img = load_image(sample_path)
variants = augment(img)
print(f"Variants: {len(variants)}  Shape: {variants[0].shape}")

Variants: 12  Shape: (96, 96, 3)


In [5]:
import h5py
from tqdm import tqdm

classes = {"open": 1, "closed": 0}

# count total samples first
total = sum(
    min(len(os.listdir(f"data/train/{c}")), 10000) * 12
    for c in classes
)
print(f"Total samples to generate: {total}")

# write directly to disk — no RAM accumulation
with h5py.File("eye_dataset_v2.h5", "w") as hf:
    X_ds = hf.create_dataset("X", shape=(total, 96, 96, 3), dtype=np.float32)
    y_ds = hf.create_dataset("y", shape=(total,), dtype=np.int32)

    idx = 0
    for label_name, label_value in classes.items():
        folder = os.path.join("data/train", label_name)
        files  = os.listdir(folder)[:10000]

        print(f"\nProcessing [{label_name}] — {len(files)} images")

        for fname in tqdm(files, desc=label_name):
            path = os.path.join(folder, fname)
            img  = load_image(path)
            if img is None:
                continue

            variants = augment(img)
            for v in variants:
                X_ds[idx] = preprocess(v)
                y_ds[idx] = label_value
                idx += 1

print(f"\nDone — saved to eye_dataset_v2.h5")
print(f"Total written: {idx} samples")

Total samples to generate: 240000

Processing [open] — 10000 images


open: 100%|██████████| 10000/10000 [00:51<00:00, 194.50it/s]



Processing [closed] — 10000 images


closed: 100%|██████████| 10000/10000 [00:48<00:00, 204.36it/s]


Done — saved to eye_dataset_v2.h5
Total written: 240000 samples


In [6]:
#train start by loading data fro, HDF5

In [8]:
import h5py
import numpy as np
from tensorflow import keras

# ─────────────────────────────────────────────────────────────
# CUSTOM HDF5 GENERATOR
# reads one batch at a time from disk
# only batch_size samples in RAM at once
# ─────────────────────────────────────────────────────────────
class HDF5Generator(keras.utils.Sequence):

    def __init__(self, h5_path, indices, batch_size=32):
        self.h5_path    = h5_path
        self.indices    = indices
        self.batch_size = batch_size

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def __getitem__(self, batch_idx):
        batch_indices = sorted(
            self.indices[batch_idx * self.batch_size :
                         (batch_idx + 1) * self.batch_size]
        )
        with h5py.File(self.h5_path, "r") as hf:
            X = hf["X"][batch_indices]
            y = hf["y"][batch_indices]
        return X, y


# ─────────────────────────────────────────────────────────────
# SPLIT INDICES — no data loaded into RAM
# ─────────────────────────────────────────────────────────────
total   = 240000
indices = np.random.permutation(total)

split        = int(total * 0.8)
train_idx    = indices[:split]
val_idx      = indices[split:]

train_gen = HDF5Generator("eye_dataset_v2.h5", train_idx, batch_size=32)
val_gen   = HDF5Generator("eye_dataset_v2.h5", val_idx,   batch_size=32)

print(f"Train batches : {len(train_gen)}")
print(f"Val   batches : {len(val_gen)}")

# test one batch loads correctly
X_batch, y_batch = train_gen[0]
print(f"Batch X shape : {X_batch.shape}")
print(f"Batch y shape : {y_batch.shape}")

I0000 00:00:1781174656.845044    4665 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Train batches : 6000
Val   batches : 1500
Batch X shape : (32, 96, 96, 3)
Batch y shape : (32,)


Input:           96 x 96 x 3
                      ↓

MobileNetV2:      3 x  3 x 1280   ← still has SPATIAL dimensions (3x3)!
                      ↓

GlobalAvgPool:        1280         ← spatial dims (3x3) COLLAPSED away
                      ↓

Dense(256):           256          ← FEATURE dim reduced
                      ↓

Dense(128):           128          ← FEATURE dim reduced
                      ↓

Dense(1):               1

In [13]:
#Now build the MobileNetV2 model
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2

# ─────────────────────────────────────────────────────────────
# STEP 1 — MOBILENETV2 BACKBONE
# pretrained on ImageNet — already knows edges, shapes, textures
# include_top=False removes ImageNet classification head
# we add our own eye open/closed head
# ─────────────────────────────────────────────────────────────

backbone = MobileNetV2(
    input_shape = (96,96,3),
    include_top=False,        # remove ImageNet head
    weights='imagenet'        # use pretrained weights
)

# freeze backbone — only train our head first
backbone.trainable = False

# ─────────────────────────────────────────────────────────────
# STEP 2 — R-CNN STYLE CLASSIFICATION HEAD
# GlobalAveragePooling = RoI pooling equivalent
# Dense layers = R-CNN classification head
# ─────────────────────────────────────────────────────────────

inputs = keras.Input(shape = (96,96,3))

x = backbone(inputs,training = False)
x = layers.GlobalAveragePooling2D()(x) #ROI pooling
x = layers.Dense(256,activation='relu')(x)
x       = layers.Dropout(0.5)(x)
x       = layers.Dense(128, activation='relu')(x)
x       = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

model = keras.Model(inputs, outputs)
model.summary()

[]


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_96             │ (None, 3, 3, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,618,945 (9.99 MB)

 Trainable params: 360,961 (1.38 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
#now compiling only the head


# ─────────────────────────────────────────────────────────────
# STEP 3 — COMPILE
# ─────────────────────────────────────────────────────────────
model.compile(
    optimizer = keras.optimizers.Adam(learning_rate = 0.001),
    loss = 'binary_crossentropy',
    metrics = ['accuracy']
)

# ─────────────────────────────────────────────────────────────
# STEP 4 — CALLBACKS
# ─────────────────────────────────────────────────────────────
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    ),
    keras.callbacks.ModelCheckpoint(
        filepath = 'eye_model_v2_best.h5',
        monitor = 'val_accuracy',
        save_best_only = True,
        verbose = 1
    )
]

# ─────────────────────────────────────────────────────────────
# STEP 5 — TRAIN PHASE 1
# backbone frozen — only head trains
# faster, stabilises head weights first
# ─────────────────────────────────────────────────────────────

print("Phase 1 — training head only...")
history_1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    callbacks=callbacks,
    verbose=1Block 1 output (32x32, before pooling):
  Each unit sees only a 3x3 patch of the RAW pixels
  → can only detect: edges, gradients, local contrast
  → physically CANNOT see anything bigger than 3x3

After MaxPool → 16x16:
  Each unit now summarizes a 2x2 area of Block1's output
  → indirectly "sees" roughly a 4x4-6x6 patch of the ORIGINAL image

Block 2 output (16x16, before its pooling):
  Each unit's 3x3 filter operates on Block1's (pooled) output
  → effective view of original image grows further

After MaxPool → 8x8:
  → each unit now "sees" roughly an 8x8-10x10 patch of original

Block 3 output (8x8, before its pooling):
  → effective view grows again

After MaxPool → 4x4:
  → each unit now "sees" a large portion of the ENTIRE 32x32 crop
)

Phase 1 — training head only...
Epoch 1/10


/home/the2003onean/Documents/deep/env/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
I0000 00:00:1781175383.504105    4665 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.


   2/6000 ━━━━━━━━━━━━━━━━━━━━ 9:58 100ms/step - accuracy: 0.4531 - loss: 0.9763

W0000 00:00:1781175386.454851    9689 cpu_allocator_impl.cc:82] Allocation of 28311552 exceeds 10% of free system memory.
W0000 00:00:1781175386.469502    9689 cpu_allocator_impl.cc:82] Allocation of 29503488 exceeds 10% of free system memory.
W0000 00:00:1781175386.554208    9699 cpu_allocator_impl.cc:82] Allocation of 28311552 exceeds 10% of free system memory.
W0000 00:00:1781175386.572507    9699 cpu_allocator_impl.cc:82] Allocation of 29503488 exceeds 10% of free system memory.
W0000 00:00:1781175386.651716    9685 cpu_allocator_impl.cc:82] Allocation of 28311552 exceeds 10% of free system memory.


6000/6000 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step - accuracy: 0.9298 - loss: 0.1803
Epoch 1: val_accuracy improved from None to 0.96021, saving model to eye_model_v2_best.h5


...


Epoch 3: finished saving model to eye_model_v2_best.h5
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 695s 116ms/step - accuracy: 0.9642 - loss: 0.0949 - val_accuracy: 0.9705 - val_loss: 0.0772
Epoch 4/10
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step - accuracy: 0.9679 - loss: 0.0859
Epoch 4: val_accuracy did not improve from 0.97052
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 687s 115ms/step - accuracy: 0.9679 - loss: 0.0858 - val_accuracy: 0.9702 - val_loss: 0.0796
Epoch 5/10
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - accuracy: 0.9704 - loss: 0.0795
Epoch 5: val_accuracy improved from 0.97052 to 0.97383, saving model to eye_model_v2_best.h5



Epoch 5: finished saving model to eye_model_v2_best.h5
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 685s 114ms/step - accuracy: 0.9698 - loss: 0.0808 - val_accuracy: 0.9738 - val_loss: 0.0683
Epoch 6/10
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - accuracy: 0.9718 - loss: 0.0765
Epoch 6: val_accuracy did not improve from 0.97383
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 690s 115ms/step - accuracy: 0.9724 - loss: 0.0757 - val_accuracy: 0.9728 - val_loss: 0.0718
Epoch 7/10
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step - accuracy: 0.9731 - loss: 0.0709
Epoch 7: val_accuracy improved from 0.97383 to 0.97479, saving model to eye_model_v2_best.h5



Epoch 7: finished saving model to eye_model_v2_best.h5
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 707s 118ms/step - accuracy: 0.9732 - loss: 0.0718 - val_accuracy: 0.9748 - val_loss: 0.0653
Epoch 8/10
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - accuracy: 0.9747 - loss: 0.0679
Epoch 8: val_accuracy improved from 0.97479 to 0.97604, saving model to eye_model_v2_best.h5



Epoch 8: finished saving model to eye_model_v2_best.h5
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 716s 119ms/step - accuracy: 0.9744 - loss: 0.0690 - val_accuracy: 0.9760 - val_loss: 0.0645
Epoch 9/10
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step - accuracy: 0.9754 - loss: 0.0657
Epoch 9: val_accuracy improved from 0.97604 to 0.97710, saving model to eye_model_v2_best.h5



Epoch 9: finished saving model to eye_model_v2_best.h5
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 723s 120ms/step - accuracy: 0.9751 - loss: 0.0664 - val_accuracy: 0.9771 - val_loss: 0.0645
Epoch 10/10
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step - accuracy: 0.9767 - loss: 0.0625
Epoch 10: val_accuracy did not improve from 0.97710
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 723s 121ms/step - accuracy: 0.9762 - loss: 0.0636 - val_accuracy: 0.9765 - val_loss: 0.0638


In [17]:
#Phase 1 complete — 97.71% validation accuracy with backbone frozen. Good baseline.

#Now Phase 2 — unfreeze backbone top layers and fine-tune:
# ─────────────────────────────────────────────────────────────
# STEP 6 — PHASE 2: FINE-TUNE BACKBONE
# unfreeze top 30 layers of MobileNetV2
# lower learning rate — don't destroy pretrained weights
# ─────────────────────────────────────────────────────────────

print(f"Total backbone layers: {len(backbone.layers)}")


# unfreeze top 30 layers only
backbone.trainable = True
for layer in backbone.layers[:-30]:
    layer.trainable = False


# count trainable vs frozen
trainable     = sum(1 for l in backbone.layers if l.trainable)
non_trainable = sum(1 for l in backbone.layers if not l.trainable)
print(f"Trainable layers    : {trainable}")
print(f"Frozen layers       : {non_trainable}")


# recompile with lower learning rate
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# ─────────────────────────────────────────────────────────────
# STEP 7 — TRAIN PHASE 2
# fine-tuning — backbone top layers + head together
# ─────────────────────────────────────────────────────────────


print("\nPhase 2 — fine-tuning top backbone layers...")
history_2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    callbacks=callbacks,
    verbose=1
)


Total backbone layers: 154
Trainable layers    : 30
Frozen layers       : 124

Phase 2 — fine-tuning top backbone layers...
Epoch 1/10
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step - accuracy: 0.9501 - loss: 0.2672
Epoch 1: val_accuracy did not improve from 0.97710
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 864s 143ms/step - accuracy: 0.9649 - loss: 0.1222 - val_accuracy: 0.9717 - val_loss: 0.0812
Epoch 2/10
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step - accuracy: 0.9833 - loss: 0.0474
Epoch 2: val_accuracy improved from 0.97710 to 0.98752, saving model to eye_model_v2_best.h5



Epoch 2: finished saving model to eye_model_v2_best.h5
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 864s 144ms/step - accuracy: 0.9841 - loss: 0.0452 - val_accuracy: 0.9875 - val_loss: 0.0359
Epoch 3/10
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step - accuracy: 0.9902 - loss: 0.0292
Epoch 3: val_accuracy improved from 0.98752 to 0.98981, saving model to eye_model_v2_best.h5



Epoch 3: finished saving model to eye_model_v2_best.h5
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 833s 139ms/step - accuracy: 0.9900 - loss: 0.0291 - val_accuracy: 0.9898 - val_loss: 0.0294
Epoch 4/10
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step - accuracy: 0.9933 - loss: 0.0196
Epoch 4: val_accuracy improved from 0.98981 to 0.99008, saving model to eye_model_v2_best.h5



Epoch 4: finished saving model to eye_model_v2_best.h5
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 860s 143ms/step - accuracy: 0.9929 - loss: 0.0205 - val_accuracy: 0.9901 - val_loss: 0.0324
Epoch 5/10
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step - accuracy: 0.9944 - loss: 0.0162
Epoch 5: val_accuracy improved from 0.99008 to 0.99037, saving model to eye_model_v2_best.h5



Epoch 5: finished saving model to eye_model_v2_best.h5
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 861s 144ms/step - accuracy: 0.9944 - loss: 0.0164 - val_accuracy: 0.9904 - val_loss: 0.0307
Epoch 6/10
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step - accuracy: 0.9960 - loss: 0.0118
Epoch 6: val_accuracy improved from 0.99037 to 0.99119, saving model to eye_model_v2_best.h5



Epoch 6: finished saving model to eye_model_v2_best.h5
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 860s 143ms/step - accuracy: 0.9956 - loss: 0.0132 - val_accuracy: 0.9912 - val_loss: 0.0359


In [1]:
import sys
print(sys.executable)

import tensorflow as tf
print(tf.__version__)

/home/the2003onean/Documents/deep/env/bin/python


I0000 00:00:1781436197.239870   21337 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


2.21.0


In [4]:
import os
import tensorflow.keras as keras

model_v2 = keras.models.load_model("eye_model_v2_best.h5")

converter = tf.lite.TFLiteConverter.from_keras_model(model_v2)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

output_path = "/home/the2003onean/Documents/deep/car/eye_model_v2.tflite"
with open(output_path, "wb") as f:
    f.write(tflite_model)

print(os.path.getsize(output_path) / 1024, "KB")

E0000 00:00:1781436274.410683   21337 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


INFO:tensorflow:Assets written to: /tmp/tmprv6p1cpl/assets


INFO:tensorflow:Assets written to: /tmp/tmprv6p1cpl/assets


Saved artifact at '/tmp/tmprv6p1cpl'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 96, 96, 3), dtype=tf.float32, name='input_layer_3')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  135746996215184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135746996215760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135746996215376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135746996217872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135746996216720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135746996218448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135746996219024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135746996219216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135746996217104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135746996217680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135746996218832

W0000 00:00:1781436280.205224   21337 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1781436280.205244   21337 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1781436280.205551   21337 reader.cc:83] Reading SavedModel from: /tmp/tmprv6p1cpl
I0000 00:00:1781436280.212726   21337 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1781436280.212738   21337 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmprv6p1cpl
I0000 00:00:1781436280.280720   21337 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1781436280.295344   21337 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1781436280.740475   21337 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmprv6p1cpl
I0000 00:00:1781436280.859118   21337 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 653577 microseconds.
I0000 00:00:1781436280.992900   2133

Both architectures create a vector the same way

V1 (custom, trained from scratch):

  Input 32x32x1 → 3 conv blocks → 4x4x128

                → GlobalAvgPool → 128-vector

                → Dropout → Dense(1, sigmoid)

V2 (transfer learning):

  Input 96x96x3 → MobileNetV2 → 3x3x1280

                → GlobalAvgPool → 1280-vector

                → Dense(256)→Dropout→Dense(128)→Dropout→Dense(1,sigmoid)

Both go: feature map → GlobalAvgPool → vector → Dense(1). The "create a vector" step is identical in both.

The REAL difference: what the vector means

V1's 128-vector:

  These 128 features were LEARNED FROM SCRATCH, specifically

  for "is this eye open or closed" — the entire network

  (including all conv layers) was trained on YOUR eye dataset.

  By the time you reach the vector, the features are ALREADY

  eye-specific. A single Dense(1) can combine them directly.

V2's 1280-vector:

  These 1280 features were learned for ImageNet — 1000 classes

  like "golden retriever", "sports car", "coffee mug".


  By the time you reach the vector, the features are GENERAL

  PURPOSE, not eye-specific. "Feature 47" might mean "furry

  texture" — useful for dogs, not obviously useful for eyelids.

Analogy:

  V1's vector = ingredients already chopped and seasoned

                for THIS exact recipe → just combine and serve


  V2's vector = a well-stocked pantry of GENERAL ingredients

                → need extra prep steps (256, 128) to turn

                  general ingredients into something specific

                  to THIS recipe, before serving

Corrected framing

"When we transfer learn, we first create a vector, unlike a

custom network" → NOT QUITE

Both create a vector the same way (GlobalAvgPool).

The difference: transfer learning's vector is GENERAL-PURPOSE

(needs extra Dense layers to specialize), while a custom

network's vector is ALREADY TASK-SPECIFIC (simple head suffices).

Quantization happens AFTER training
It takes the already-trained weights from eye_model_v2_best.h5
and converts them from float32 to INT8

Training:      learns WHAT the weights should be
Quantization:  compresses HOW those weights are stored

The 100 sample "representative dataset" is NOT for training
— it's just used to measure the range of values in each
layer so the converter knows how to scale them to INT8
without losing too much precision

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import h5py

model_v2 = keras.models.load_model("eye_model_v2_best.h5")

# ── representative dataset for calibration ──────────────────
# quantization needs sample data to calibrate INT8 ranges
def representative_data_gen():
    with h5py.File("eye_dataset_v2.h5", "r") as hf:
        # take 100 random samples
        indices = np.random.choice(240000, 100, replace=False)
        indices = sorted(indices)
        for i in indices:
            img = hf["X"][i]
            yield [img.reshape(1, 96, 96, 3).astype(np.float32)]

# ── convert with INT8 quantization ──────────────────────────
converter = tf.lite.TFLiteConverter.from_keras_model(model_v2)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_f16 = converter.convert()

output_path = "/home/the2003onean/Documents/deep/car/eye_model_v2_f16.tflite"
with open(output_path, "wb") as f:
    f.write(tflite_f16)

import os
original = os.path.getsize("eye_model_v2.tflite") / 1024
f16_size  = os.path.getsize(output_path) / 1024
print(f"Original V2 (float32): {original:.1f} KB")
print(f"Float16 V2:            {f16_size:.1f} KB")
print(f"Size reduction:        {(1 - f16_size/original)*100:.1f}%")

INFO:tensorflow:Assets written to: /tmp/tmpnwvkqd_z/assets


INFO:tensorflow:Assets written to: /tmp/tmpnwvkqd_z/assets


Saved artifact at '/tmp/tmpnwvkqd_z'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 96, 96, 3), dtype=tf.float32, name='input_layer_3')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  135007673251216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135007673252752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135007673252944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135007673253712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135007673251024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135007673251984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135007673252560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135007673252368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135007673252176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135007673250064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135007673250640

W0000 00:00:1781621927.195824   11573 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1781621927.195849   11573 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1781621927.196013   11573 reader.cc:83] Reading SavedModel from: /tmp/tmpnwvkqd_z
I0000 00:00:1781621927.203738   11573 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1781621927.203771   11573 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpnwvkqd_z
I0000 00:00:1781621927.297645   11573 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1781621927.765757   11573 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpnwvkqd_z
I0000 00:00:1781621927.908111   11573 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 712108 microseconds.
